# style-lora — Together.ai로 학습 (GPU 런타임 필요 없음)

이 노트북은 로컬/Colab GPU로 직접 학습하는 대신, **Together.ai 클라우드에
학습을 맡기고 진행 상황만 확인**하는 버전입니다. 런타임은 GPU 없이
**CPU(기본값)**로 둬도 됩니다 — 실제 학습은 Together 서버에서 돌아갑니다.

**절대 이 txt 원문을 공개 저장소에 커밋하지 마세요.**

In [ ]:
from getpass import getpass
gh_token = getpass('GitHub 토큰 입력 (저장소가 private이라 필요): ')
!rm -rf repo
!git clone -b claude/novelai-custom-module-limits-a4uncu https://{gh_token}@github.com/yeou77/-.git repo
%cd repo/style-lora
!pip install -q -r requirements.txt

## 1) 소설 통째로 업로드

In [ ]:
from google.colab import files
uploaded = files.upload()  # 소설 txt 2개 그대로 선택
import shutil, os
os.makedirs('data/raw', exist_ok=True)
for name in uploaded:
    shutil.move(name, f'data/raw/{name}')
print(os.listdir('data/raw'))

## 2) 자동 전처리

In [ ]:
!python scripts/preprocess.py auto

## 3) Together API 키 입력 + 학습 시작 + 진행 확인 + 다운로드까지 한 번에
이 셀 하나가 업로드 → 파인튜닝 job 생성 → 완료까지 대기 → 결과 다운로드를
전부 처리합니다. 완료될 때까지 수 분~수십 분 걸릴 수 있습니다.

In [ ]:
from getpass import getpass
together_key = getpass('Together API 키 입력: ')
with open('together_key.txt', 'w') as f:
    f.write(together_key)

!python scripts/train_together.py --api-key-file together_key.txt \
    --model Qwen/Qwen3.5-9B \
    --download-to outputs/style-lora-together.tar.zst

## 4) 결과를 Drive에 백업

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp outputs/style-lora-together.tar.zst /content/drive/MyDrive/style-lora-together.tar.zst